In [1]:
import os
import sys

import pickle

import torch

import pandas as pd

import pm4py

from config.feature_config import FeatureConfig
from config.ga_config import GAConfig

from utils.general_utils import set_stdout_to_file, set_seed
from utils.feature_utils import df_to_sequence_array

from model.preprocessor import PreprocessorArtifacts

from model.next_event_model import ProcessLSTM
from model.model_wrapper import ModelWrapper

from process.engine import ProcessModelConstraintEngine
from process.constraint import ProcessConstraint

from ga_search.search import CounterfactualGA

### --- Load Dataset & Models ---

In [2]:
set_seed(seed=42)

In [3]:
df = pd.read_excel(
    "../../../data/bpic12.xlsx",
    engine="openpyxl",
    keep_default_na=False,
    dtype={
        "case:concept:name": "string",
        "concept:name": "string",
        "lifecycle:transition": "string",
        "org:resource": "string",
        "case:REG_DATE_HR": "string",
        "case:REG_DATE_DAY": "string",
        "case:REG_DATE_MON": "string",
        "case:AMOUNT_REQ": "float32",
        "time_delta": "float32",
    }
)

df["time:timestamp"] = pd.to_datetime(df["time:timestamp"])

In [4]:
df.head(20)

,case:concept:name,time:timestamp,case:AMOUNT_REQ,case:REG_DATE_DAY,case:REG_DATE_HR,case:REG_DATE_MON,concept:name,lifecycle:transition,org:resource,time_delta
0,173688,2011-10-01 00:38:44.546,20000.0,Saturday,12 AM,October,A_SUBMITTED,COMPLETE,112,0.000000
1,173688,2011-10-01 00:38:44.880,20000.0,Saturday,12 AM,October,A_PARTLYSUBMITTED,COMPLETE,112,0.334000
2,173688,2011-10-01 00:39:37.906,20000.0,Saturday,12 AM,October,A_PREACCEPTED,COMPLETE,112,53.026001
3,173688,2011-10-01 11:42:43.308,20000.0,Saturday,12 AM,October,A_ACCEPTED,COMPLETE,10862,39785.402344
4,173688,2011-10-01 11:45:09.243,20000.0,Saturday,12 AM,October,O_SELECTED,COMPLETE,10862,145.934998
5,173688,2011-10-01 11:45:09.243,20000.0,Saturday,12 AM,October,A_FINALIZED,COMPLETE,10862,0.000000
6,173688,2011-10-01 11:45:11.197,20000.0,Saturday,12 AM,October,O_CREATED,COMPLETE,10862,1.954000
7,173688,2011-10-01 11:45:11.380,20000.0,Saturday,12 AM,October,O_SENT,COMPLETE,10862,0.183000
8,173688,2011-10-10 11:33:03.668,20000.0,Saturday,12 AM,October,O_SENT_BACK,COMPLETE,11049,776872.312500
9,173688,2011-10-13 10:37:29.226,20000.0,Saturday,12 AM,October,A_REGISTERED,COMPLETE,10629,255865.562500


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [6]:
feature_config = FeatureConfig.load(
    path = "../pretrained_models/"
)

In [7]:
feature_config.summary()

  activity_feature:    concept:name
  feature_order:       ['case:AMOUNT_REQ', 'case:REG_DATE_DAY', 'case:REG_DATE_HR', 'case:REG_DATE_MON', 'concept:name', 'lifecycle:transition', 'org:resource', 'time_delta']
--------------------------------------------------------------------------------------------------------------------------------------
Feature                        Type           Level    Vary   Range/Categories                         MAD        Source              
--------------------------------------------------------------------------------------------------------------------------------------
time_delta                     continuous     event    yes    [0.00, 1315.95]                          0.4980     quantile_derived    
case:AMOUNT_REQ                continuous     case     yes    [3000.00, 40000.00]                      5000.0000  quantile_derived    
case:REG_DATE_DAY              categorical    case     yes    ['Friday', 'Monday', 'Saturday', ...]    N/A        

In [8]:
preprocessor_artifacts = PreprocessorArtifacts.load(
    path = "../pretrained_models/"
)

In [9]:
model = ProcessLSTM.load(
    path = "../pretrained_models/"
)

In [10]:
model_wrapper = ModelWrapper(
    model=model,
    preprocessor_artifacts=preprocessor_artifacts,
    device=device
)

### --- Process Constraints ---

In [11]:
engine = ProcessModelConstraintEngine.load(
    path = "../pretrained_models/"
)

In [12]:
engine.parallel_sets

[{'A_FINALIZED', 'O_SELECTED'},
 {'A_DECLINED', 'O_DECLINED'},
 {'A_ACTIVATED', 'A_APPROVED', 'A_REGISTERED', 'O_ACCEPTED'}]

In [13]:
engine.branching_sets

[{'A_CANCELLED',
  'A_FINALIZED',
  'O_CREATED',
  'O_SELECTED',
  'O_SENT',
  'O_SENT_BACK'},
 {'A_ACTIVATED',
  'A_APPROVED',
  'A_DECLINED',
  'A_REGISTERED',
  'O_ACCEPTED',
  'O_DECLINED'}]

### --- Counterfactuals ---

In [14]:
ga_config_equal = GAConfig(
    w_distance=1.0,
    w_sparsity=1.0,
    w_margin=1.0,
    w_process_violation=1.0,
)
ga_config_equal.validate()

ga_config_weighted = GAConfig(
    w_distance=1.0,
    w_sparsity=1.0,
    w_margin=3.0,
    w_process_violation=3.0
)
ga_config_weighted.validate()

In [15]:
log_file, original_stdout = set_stdout_to_file(filepath="logs/bpic12-cf_domain_examples_ga_output.txt", console=False)

In [16]:
with open("../experiments/cf_domain_examples.pkl", "rb") as f:
    data = pickle.load(f)

experiments = data["experiments"]
metadata = data["metadata"]

In [17]:
def ga_search_for_experiment(
    engine,
    experiment,
    exp_name,
    ga_config,
    train_arr,
    change_allowed_positions,
    activity_idx,
    user_rules
):
    results = []

    candidate_groups = experiment["candidates"]
    conf_arr = experiment["conformant"]

    for group_idx, cand_arr in enumerate(candidate_groups):

        print(
            f"Running experiment: {exp_name} "
            f"(group {group_idx + 1}) | cases: {cand_arr.shape[0]}"
        )

        for i in range(cand_arr.shape[0]):

            trace = cand_arr[i]
            trace_activities = [row[activity_idx] for row in trace]
            
            flexible_constraints = engine.generate_flexible_constraints(trace_activities)
            print("Flexible Constraints:", flexible_constraints)

            desired_constraints = engine.generate_desired_constraints(
                trace_activities=trace_activities,
                user_specs=user_rules
            )
            print("Desired Constraints: ", desired_constraints)

            cf_GA = CounterfactualGA(
                ga_config=ga_config,
                feature_config=feature_config,
                model_wrapper=model_wrapper
            )

            cf = cf_GA.search(
                trace=trace,
                train_arr=train_arr,
                conformant_arr=conf_arr,
                change_allowed_positions=change_allowed_positions,
                desired_constraints=desired_constraints,
                flexible_constraints=flexible_constraints,
                output_path=(
                    f"./counterfactuals/{exp_name}_group_{group_idx + 1}_case_{i}.xlsx"
                )
            )

            results.append(cf["eval"])

    return results

#### 1. A_ACCEPTED Milestone

In [18]:
experiment_a_preaccepted = experiments["a_preaccepted"]

change_allowed_positions = [0, 1]
user_rules_a_preaccepted = {
    ("A_DECLINED", 1): {"A_PREACCEPTED"},
    ("A_CANCELLED", 1): {"A_PREACCEPTED"},
}

train_arr_a_preaccepted = df_to_sequence_array(
    df=df,
    case_id_field="case:concept:name",
    feature_config=feature_config,
    prefix_len=metadata["prefix_lengths"]["a_preaccepted"],
    sort_field="time:timestamp"
)

results_a_preaccepted = ga_search_for_experiment(
    engine=engine,
    experiment=experiment_a_preaccepted,
    exp_name="a_preaccepted",
    ga_config=ga_config_equal,
    change_allowed_positions=change_allowed_positions,
    user_rules=user_rules_a_preaccepted,
    activity_idx=feature_config.feature_to_idx[feature_config.activity_feature],
    train_arr=train_arr_a_preaccepted
)

In [19]:
experiment_a_accepted = experiments["a_accepted"]

change_allowed_positions = [0, 1, 2]
user_rules_a_accepted = {
    ("A_DECLINED", 1): {"A_ACCEPTED"},
    ("A_CANCELLED", 1): {"A_ACCEPTED"},
}

train_arr_a_accepted = df_to_sequence_array(
    df=df,
    case_id_field="case:concept:name",
    feature_config=feature_config,
    prefix_len=metadata["prefix_lengths"]["a_accepted"],
    sort_field="time:timestamp"
)

results_a_accepted = ga_search_for_experiment(
    engine=engine,
    experiment=experiment_a_accepted,
    exp_name="a_accepted",
    ga_config=ga_config_equal,
    change_allowed_positions=change_allowed_positions,
    user_rules=user_rules_a_accepted,
    activity_idx=feature_config.feature_to_idx[feature_config.activity_feature],
    train_arr=train_arr_a_accepted
)

#### 2. A_FINALIZED Milestone

In [20]:
experiment_a_finalized = experiments["a_finalized"]

change_allowed_positions = [0, 1, 2, 3]
user_rules_a_finalized = {
    ("A_DECLINED", 1): {"A_FINALIZED", "O_SELECTED"},
    ("A_CANCELLED", 1): {"A_FINALIZED", "O_SELECTED"},
}

train_arr_a_finalized = df_to_sequence_array(
    df=df,
    case_id_field="case:concept:name",
    feature_config=feature_config,
    prefix_len=metadata["prefix_lengths"]["a_finalized"],
    sort_field="time:timestamp"
)

results_a_finalized = ga_search_for_experiment(
    engine=engine,
    experiment=experiment_a_finalized,
    exp_name="a_finalized",
    ga_config=ga_config_equal,
    change_allowed_positions=change_allowed_positions,
    user_rules=user_rules_a_finalized,
    activity_idx=feature_config.feature_to_idx[feature_config.activity_feature],
    train_arr=train_arr_a_finalized
)

#### 3. A_APPROVED Milestone

In [21]:
experiment_1_a_approved = experiments["1_a_approved"]

change_allowed_positions = [0, 1, 2, 3, 4, 5, 6, 7]
user_rules_1_a_approved = {
    ("A_CANCELLED", 1): {"O_SENT_BACK"},
    ("O_CANCELLED", 1): {"A_APPROVED", "O_ACCEPTED"},
}

train_arr_1_a_approved = df_to_sequence_array(
    df=df,
    case_id_field="case:concept:name",
    feature_config=feature_config,
    prefix_len=metadata["prefix_lengths"]["1_a_approved"],
    sort_field="time:timestamp"
)

results_1_a_approved_equal = ga_search_for_experiment(
    engine=engine,
    experiment=experiment_1_a_approved,
    exp_name="1_a_approved_equal",
    ga_config=ga_config_equal,
    change_allowed_positions=change_allowed_positions,
    user_rules=user_rules_1_a_approved,
    activity_idx=feature_config.feature_to_idx[feature_config.activity_feature],
    train_arr=train_arr_1_a_approved
)
results_1_a_approved_weighted = ga_search_for_experiment(
    engine=engine,
    experiment=experiment_1_a_approved,
    exp_name="1_a_approved_weighted",
    ga_config=ga_config_weighted,
    change_allowed_positions=change_allowed_positions,
    user_rules=user_rules_1_a_approved,
    activity_idx=feature_config.feature_to_idx[feature_config.activity_feature],
    train_arr=train_arr_1_a_approved
)

experiment_2_a_approved = experiments["2_a_approved"]

change_allowed_positions = [0, 1, 2, 3, 4, 5, 6, 7, 8]
user_rules_2_a_approved = {
    ("A_DECLINED", 1): {"A_APPROVED"},
    ("O_DECLINED", 1): {"O_ACCEPTED"},
}

train_arr_2_a_approved = df_to_sequence_array(
    df=df,
    case_id_field="case:concept:name",
    feature_config=feature_config,
    prefix_len=metadata["prefix_lengths"]["2_a_approved"],
    sort_field="time:timestamp"
)

results_2_a_approved_equal = ga_search_for_experiment(
    engine=engine,
    experiment=experiment_2_a_approved,
    exp_name="2_a_approved_equal",
    ga_config=ga_config_equal,
    change_allowed_positions=change_allowed_positions,
    user_rules=user_rules_2_a_approved,
    activity_idx=feature_config.feature_to_idx[feature_config.activity_feature],
    train_arr=train_arr_2_a_approved
)
results_2_a_approved_weighted = ga_search_for_experiment(
    engine=engine,
    experiment=experiment_2_a_approved,
    exp_name="2_a_approved_weighted",
    ga_config=ga_config_weighted,
    change_allowed_positions=change_allowed_positions,
    user_rules=user_rules_2_a_approved,
    activity_idx=feature_config.feature_to_idx[feature_config.activity_feature],
    train_arr=train_arr_2_a_approved
)

#### 4. Swapped Ordering of A_APPROVED A_REGISTERED

In [22]:
change_allowed_positions = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

experiment_1_swap = experiments["1_swap"]

user_rules_1_swap = {
    ("A_REGISTERED", 1): {"A_APPROVED"},
    ("A_APPROVED", 1): {"A_REGISTERED"},
}

train_arr_1_swap = df_to_sequence_array(
    df=df,
    case_id_field="case:concept:name",
    feature_config=feature_config,
    prefix_len=metadata["prefix_lengths"]["1_swap"],
    sort_field="time:timestamp"
)

results_1_swap_equal = ga_search_for_experiment(
    engine=engine,
    experiment=experiment_1_swap,
    exp_name="1_swap_equal",
    ga_config=ga_config_equal,
    change_allowed_positions=change_allowed_positions,
    user_rules=user_rules_1_swap,
    activity_idx=feature_config.feature_to_idx[feature_config.activity_feature],
    train_arr=train_arr_1_swap
)
results_1_swap_weighted = ga_search_for_experiment(
    engine=engine,
    experiment=experiments["1_swap"],
    exp_name="1_swap_weighted",
    ga_config=ga_config_weighted,
    change_allowed_positions=change_allowed_positions,
    user_rules=user_rules_1_swap,
    activity_idx=feature_config.feature_to_idx[feature_config.activity_feature],
    train_arr=train_arr_1_swap
)

experiment_2_swap = experiments["2_swap"]

user_rules_2_swap = {
    ("A_ACTIVATED", 1): {"A_APPROVED"},
    ("A_APPROVED", 1): {"A_ACTIVATED"},
}

train_arr_2_swap = df_to_sequence_array(
    df=df,
    case_id_field="case:concept:name",
    feature_config=feature_config,
    prefix_len=metadata["prefix_lengths"]["2_swap"],
    sort_field="time:timestamp"
)

results_2_swap_equal = ga_search_for_experiment(
    engine=engine,
    experiment=experiment_2_swap,
    exp_name="2_swap_equal",
    ga_config=ga_config_equal,
    change_allowed_positions=change_allowed_positions,
    user_rules=user_rules_2_swap,
    activity_idx=feature_config.feature_to_idx[feature_config.activity_feature],
    train_arr=train_arr_2_swap
)
results_2_swap_weighted = ga_search_for_experiment(
    engine=engine,
    experiment=experiment_2_swap,
    exp_name="2_swap_weighted",
    ga_config=ga_config_weighted,
    change_allowed_positions=change_allowed_positions,
    user_rules=user_rules_2_swap,
    activity_idx=feature_config.feature_to_idx[feature_config.activity_feature],
    train_arr=train_arr_2_swap
)

experiment_3_swap = experiments["3_swap"]

user_rules_3_swap = {
    ("A_REGISTERED", 1): {"A_APPROVED"},
    ("A_APPROVED", 1): {"A_REGISTERED"},
}

train_arr_3_swap = df_to_sequence_array(
    df=df,
    case_id_field="case:concept:name",
    feature_config=feature_config,
    prefix_len=metadata["prefix_lengths"]["3_swap"],
    sort_field="time:timestamp"
)

results_3_swap_equal = ga_search_for_experiment(
    engine=engine,
    experiment=experiment_3_swap,
    exp_name="3_swap_equal",
    ga_config=ga_config_equal,
    change_allowed_positions=change_allowed_positions,
    user_rules=user_rules_3_swap,
    activity_idx=feature_config.feature_to_idx[feature_config.activity_feature],
    train_arr=train_arr_3_swap
)
results_3_swap_weighted = ga_search_for_experiment(
    engine=engine,
    experiment=experiment_3_swap,
    exp_name="3_swap_weighted",
    ga_config=ga_config_weighted,
    change_allowed_positions=change_allowed_positions,
    user_rules=user_rules_3_swap,
    activity_idx=feature_config.feature_to_idx[feature_config.activity_feature],
    train_arr=train_arr_3_swap
)

experiment_4_swap = experiments["4_swap"]

user_rules_4_swap = {
    ("A_ACTIVATED", 1): {"A_APPROVED"},
    ("A_REGISTERED", 1): {"A_ACTIVATED"},
    ("A_APPROVED", 1): {"A_REGISTERED"},
}

train_arr_4_swap = df_to_sequence_array(
    df=df,
    case_id_field="case:concept:name",
    feature_config=feature_config,
    prefix_len=metadata["prefix_lengths"]["4_swap"],
    sort_field="time:timestamp"
)

results_4_swap_equal = ga_search_for_experiment(
    engine=engine,
    experiment=experiment_4_swap,
    exp_name="4_swap_equal",
    ga_config=ga_config_equal,
    change_allowed_positions=change_allowed_positions,
    user_rules=user_rules_4_swap,
    activity_idx=feature_config.feature_to_idx[feature_config.activity_feature],
    train_arr=train_arr_4_swap
)
results_4_swap_weighted = ga_search_for_experiment(
    engine=engine,
    experiment=experiment_4_swap,
    exp_name="4_swap_weighted",
    ga_config=ga_config_weighted,
    change_allowed_positions=change_allowed_positions,
    user_rules=user_rules_4_swap,
    activity_idx=feature_config.feature_to_idx[feature_config.activity_feature],
    train_arr=train_arr_4_swap
)

### --- Overall Evaluation ---

In [23]:
def calculate_summary_eval(overall_results):
    df = pd.DataFrame(overall_results)

    summary_evaluation_df = pd.DataFrame({
        "Metric": df.columns,
        "Mean": df.mean(numeric_only=True).values,
        "Std": df.std(numeric_only=True).values,
        "Min": df.min(numeric_only=True).values,
        "Max": df.max(numeric_only=True).values,
    })

    return summary_evaluation_df


def analyze_overall_results(overall_results, group_name: str):

    os.makedirs("domain_examples_results", exist_ok=True)

    # Flatten experiments
    all_results = [
        result
        for experiment in overall_results
        for result in experiment
    ]

    summary_eval_df = calculate_summary_eval(all_results)

    with pd.ExcelWriter(
        f"domain_examples_results/evaluation_{group_name}.xlsx",
        engine="openpyxl"
    ) as writer:

        summary_eval_df.to_excel(
            writer,
            sheet_name="summary_evaluation",
            index=False
        )

    return summary_eval_df

In [24]:
analyze_overall_results([results_a_preaccepted], group_name="a_preaccepted")

,Metric,Mean,Std,Min,Max
0,DISTANCE,0.478581,0.047292,0.408776,0.553013
1,CONT_DISTANCE,0.470798,0.071572,0.370919,0.587084
2,CAT_DISTANCE,0.486364,0.067271,0.340000,0.590000
3,SPARSITY,0.571591,0.065214,0.468750,0.662500
4,PROCESS_VIOLATION,0.000000,0.000000,0.000000,0.000000
5,MARGIN_LOSS,0.048461,0.030748,0.021067,0.122673
6,MARGIN_LOSS_NA,0.048461,0.030748,0.021067,0.122673
7,MARGIN_LOSS_FLIPPED,0.000000,0.000000,0.000000,0.000000
8,MARGIN_LOSS_NA_FLIPPED,0.000000,0.000000,0.000000,0.000000
9,FITNESS,1.098632,0.108232,0.911925,1.267637


In [25]:
analyze_overall_results([results_a_accepted], group_name="a_accepted")

,Metric,Mean,Std,Min,Max
0,DISTANCE,0.494177,0.044347,0.377227,0.570092
1,CONT_DISTANCE,0.435020,0.074497,0.302450,0.563796
2,CAT_DISTANCE,0.553333,0.050898,0.441667,0.650000
3,SPARSITY,0.586000,0.051006,0.435000,0.655000
4,PROCESS_VIOLATION,0.000909,0.002798,0.000000,0.009091
5,MARGIN_LOSS,0.006361,0.013034,0.000000,0.043127
6,MARGIN_LOSS_NA,0.240975,0.294042,0.000000,0.706795
7,MARGIN_LOSS_FLIPPED,0.000000,0.000000,0.000000,0.000000
8,MARGIN_LOSS_NA_FLIPPED,0.304999,0.392662,0.000000,0.949997
9,FITNESS,1.087447,0.085717,0.848315,1.218585


In [26]:
analyze_overall_results([results_a_finalized], group_name="a_finalized")

,Metric,Mean,Std,Min,Max
0,DISTANCE,0.485212,0.049101,0.388521,0.608212
1,CONT_DISTANCE,0.419353,0.081633,0.277042,0.587853
2,CAT_DISTANCE,0.551071,0.054609,0.471429,0.671429
3,SPARSITY,0.568333,0.053827,0.445833,0.670833
4,PROCESS_VIOLATION,0.004615,0.013985,0.000000,0.061538
5,MARGIN_LOSS,0.000409,0.001260,0.000000,0.004336
6,MARGIN_LOSS_NA,0.004578,0.019589,0.000000,0.087724
7,MARGIN_LOSS_FLIPPED,0.000000,0.000000,0.000000,0.000000
8,MARGIN_LOSS_NA_FLIPPED,0.007500,0.033541,0.000000,0.149999
9,FITNESS,1.058570,0.093691,0.900229,1.286738


In [27]:
analyze_overall_results([results_1_a_approved_equal, results_2_a_approved_equal], group_name="a_approved_equal")

,Metric,Mean,Std,Min,Max
0,DISTANCE,0.457987,0.057861,0.289194,0.601476
1,CONT_DISTANCE,0.372271,0.079626,0.114751,0.530747
2,CAT_DISTANCE,0.543703,0.071757,0.381818,0.704167
3,SPARSITY,0.535159,0.065813,0.352500,0.704545
4,PROCESS_VIOLATION,0.099348,0.175157,0.000000,0.500000
5,MARGIN_LOSS,0.225686,0.186873,0.000000,0.641107
6,MARGIN_LOSS_NA,0.351106,0.356234,0.000000,0.957967
7,MARGIN_LOSS_FLIPPED,0.131875,0.231464,0.000000,0.675000
8,MARGIN_LOSS_NA_FLIPPED,0.271562,0.430663,0.000000,1.000000
9,FITNESS,1.318179,0.308091,0.719144,1.992313


In [28]:
analyze_overall_results([results_1_a_approved_weighted, results_2_a_approved_weighted], group_name="a_approved_weighted")

,Metric,Mean,Std,Min,Max
0,DISTANCE,0.592162,0.055139,0.466602,0.722429
1,CONT_DISTANCE,0.503282,0.084197,0.301003,0.766792
2,CAT_DISTANCE,0.681042,0.063099,0.512500,0.858333
3,SPARSITY,0.692500,0.060946,0.568182,0.831818
4,PROCESS_VIOLATION,0.082771,0.148589,0.000000,0.496154
5,MARGIN_LOSS,0.215342,0.177234,0.000000,0.506160
6,MARGIN_LOSS_NA,0.357120,0.355909,0.000000,0.957466
7,MARGIN_LOSS_FLIPPED,0.125312,0.218434,0.000000,0.525000
8,MARGIN_LOSS_NA_FLIPPED,0.279375,0.427307,0.000000,1.000000
9,FITNESS,2.179001,0.960546,1.105206,4.184588


In [29]:
analyze_overall_results([results_1_swap_equal, results_2_swap_equal, results_3_swap_equal, results_4_swap_equal], group_name="swap_equal")

,Metric,Mean,Std,Min,Max
0,DISTANCE,0.526654,0.055990,0.378716,0.620901
1,CONT_DISTANCE,0.452923,0.085916,0.295893,0.645648
2,CAT_DISTANCE,0.600385,0.059290,0.461538,0.703846
3,SPARSITY,0.609896,0.062573,0.450000,0.702083
4,PROCESS_VIOLATION,0.027883,0.068107,0.000000,0.356250
5,MARGIN_LOSS,0.077122,0.111180,0.000000,0.327382
6,MARGIN_LOSS_NA,0.383599,0.208592,0.029228,0.771064
7,MARGIN_LOSS_FLIPPED,0.059999,0.147063,0.000000,0.499977
8,MARGIN_LOSS_NA_FLIPPED,0.402915,0.237316,0.000000,0.999997
9,FITNESS,1.241554,0.157860,0.840589,1.675031


In [30]:
analyze_overall_results([results_1_swap_weighted, results_2_swap_weighted, results_3_swap_weighted, results_4_swap_weighted], group_name="swap_weighted")

,Metric,Mean,Std,Min,Max
0,DISTANCE,0.645688,0.060156,0.520203,0.771006
1,CONT_DISTANCE,0.561280,0.084388,0.418880,0.761243
2,CAT_DISTANCE,0.730096,0.057596,0.600000,0.842308
3,SPARSITY,0.748958,0.073369,0.595833,0.862500
4,PROCESS_VIOLATION,0.051152,0.130460,0.000000,0.475000
5,MARGIN_LOSS,0.023443,0.066240,0.000000,0.275772
6,MARGIN_LOSS_NA,0.345939,0.180776,0.000000,0.725387
7,MARGIN_LOSS_FLIPPED,0.027499,0.098838,0.000000,0.474995
8,MARGIN_LOSS_NA_FLIPPED,0.390624,0.216403,0.000000,0.924995
9,FITNESS,1.618431,0.531810,1.124607,2.996143


### --- Cleanup ---

In [31]:
sys.stdout = original_stdout
log_file.close()